In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp


import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd
import gc

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *
from analysis_village.cc1pi.var_configs import *



from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# Load DataFrame MC

In [ ]:
from cols_to_keep import *

#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file

bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst_pruned.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100, reprocess_df = False, reprocess_truth = False)
mc_bnb_evt_df = mc_bnb_df['cc1pi']
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']
'''
cols_to_keep = [
    ('truth','nu_categ', '', '', '',''),
    ('truth','genie_categ', '', '', '',''),
    ('truth','nu_categ_proton_reduced', '', '', '','')
]
mc_bnb_nu_df = mc_bnb_nu_df[cols_to_keep]
'''
reco_cols_to_keep =  min_reco_cols_to_keep + [
    ('truth','nu_categ', '', '', '',''),
    ('truth','genie_categ', '', '', '',''),
    ('truth','nu_categ_proton_reduced', '', '', '','')
]

mc_bnb_evt_df[reco_cols_to_keep]

'''

keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
#load in time cosmic df
mc_in_time_cosmics_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_in_time_cosmics.df", keys2load, 100)
mc_in_time_cosmics_evt_df = mc_in_time_cosmics_df['cc1pi']
mc_in_time_cosmics_hdr_df = mc_in_time_cosmics_df['hdr']

mc_in_time_cosmics_evt_df = mc_in_time_cosmics_evt_df[min_reco_cols_to_keep]

cols_to_keep_truth = [
    ('nu_categ', '', '', ''),
    ('genie_categ', '', '', ''),
    ('nu_categ_proton_reduced', '', '', '')
]


#Load CV lowE dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_lowE_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_1e20_lowE_CV.df", keys2load, 1, filter_df = False)
mc_bnb_lowE_evt_df = mc_bnb_lowE_df['cc1pi']
mc_bnb_lowE_nu_df = mc_bnb_lowE_df['nudf']
mc_bnb_lowE_hdr_df = mc_bnb_lowE_df['hdr']


mc_bnb_lowE_evt_df = mc_bnb_lowE_evt_df[min_reco_cols_to_keep]
mc_bnb_lowE_nu_df = mc_bnb_lowE_nu_df[cols_to_keep_truth]


'''

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_fixdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_hdr_df = data_df['hdr']

#load off beam light df
off_beam_light_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_offbeamlight.df", keys2load, 100)
off_beam_light_evt_df = off_beam_light_df['cc1pi']
off_beam_light_hdr_df = off_beam_light_df['hdr']

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')
# BNB data
data_tot_pot = data_hdr_df['pot'].sum()
print("data_tot_pot: %.3e" %(data_tot_pot))
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))
data_gates = data_hdr_df.nbnbinfo.sum()
print("data tot gates : %.3e" %(data_gates))

# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))


'''
#Low E
mc_bnb_lowE_tot_pot = mc_bnb_lowE_hdr_df['pot'].sum()
print("dirt_tot_pot: %.3e" %(mc_bnb_lowE_tot_pot))
mc_bnb_lowE_pot_scale = data_tot_pot / mc_bnb_lowE_tot_pot
print("dirt_pot_scale: %.3e" %(mc_bnb_lowE_pot_scale))
mc_bnb_lowE_evt_df[pot_weight_col] = mc_bnb_lowE_pot_scale * np.ones(len(mc_bnb_lowE_evt_df))




intime_gates = mc_in_time_cosmics_hdr_df[mc_in_time_cosmics_hdr_df['first_in_subrun'] == 1]['ngenevt'].sum()
print("intime cosmics data gates: {:.2e}".format(intime_gates))
f = 0.075
scale_intime_to_lightdata = (1-f)*data_gates/intime_gates
print("goal scale: {:.2f}".format(scale_intime_to_lightdata))
mc_in_time_cosmics_evt_df[pot_weight_col] = scale_intime_to_lightdata * np.ones(len(mc_in_time_cosmics_evt_df))
'''


off_beam_data_gates = off_beam_light_hdr_df.noffbeambnb.sum()
print("intime cosmics data gates: {:.2e}".format(off_beam_data_gates))
f = 0.075
scale_off_beam_to_lightdata = (1-f)*data_gates/off_beam_data_gates
print("goal scale: {:.2f}".format(scale_off_beam_to_lightdata))
off_beam_light_evt_df[pot_weight_col] = scale_off_beam_to_lightdata * np.ones(len(off_beam_light_evt_df))



In [ ]:
#Add MC stat:
mc_evt_df = mc_bnb_evt_df
'''
if "ar23p" in bnb_path:
    mc_evt_df = perform_truth_matching_low_memmory(mc_evt_df, mc_bnb_nu_df)
else:
    mc_evt_df = perform_truth_matching(mc_evt_df, mc_bnb_nu_df)   
print("Finished loading")    
'''


'''
mc_in_time_cosmics_evt_df = perform_truth_matching(mc_in_time_cosmics_evt_df, mc_bnb_lowE_nu_df.head(1))   
print("TM for cosmic done")

mc_in_time_cosmics_evt_df =mc_in_time_cosmics_evt_df.groupby(['__ntuple', 'entry', 'rec.slc..index']).first()
mc_bnb_lowE_evt_df = perform_truth_matching(mc_bnb_lowE_evt_df, mc_bnb_lowE_nu_df)
mc_bnb_lowE_evt_df =mc_bnb_lowE_evt_df.groupby(['__ntuple', 'entry', 'rec.slc..index']).first()


mc_evt_df = concat_shift_first_index(mc_evt_df,mc_in_time_cosmics_evt_df)
mc_evt_df = concat_shift_first_index(mc_evt_df,mc_bnb_lowE_evt_df)
'''

# Test background composition

In [ ]:
mc_cumulative_masks = build_event_cumulative_masks(mc_evt_df, sideband = "")
data_cumulative_masks = build_event_cumulative_masks(data_evt_df, sideband = "")

In [ ]:
for name in mc_cumulative_masks.keys():
    # Retrieve the pre-calculated cumulative masks
    mc_mask = mc_cumulative_masks[name]
    data_mask = data_cumulative_masks[name]
    
    # Filter dataframes and count unique events
    n_mc = get_n_evt(mc_evt_df[mc_mask ], True)
    n_data = get_n_evt(data_evt_df[data_mask], False)
    
    print(f"{name:<15} | {n_mc:<12} | {n_data:<12}")


In [ ]:
print(get_n_evt(data_evt_df[data_cumulative_masks["energy"] & mask_dict["0p"](data_evt_df)], False))
print(get_n_evt(data_evt_df[data_cumulative_masks["energy"] & mask_dict["1p"](data_evt_df)], False))
print(get_n_evt(data_evt_df[data_cumulative_masks["energy"] & mask_dict["2plusp"](data_evt_df)], False))

In [ ]:
def get_evts(df, var_col, bins=None, verbose=True):
    var = df[var_col]
    if ('slc','wgt','','','','') in df.columns:
        weights = df.loc[:, ('slc','wgt','','','','')]
    else:
        if verbose:
            print("No pot_weight column found, returning 1 as weight (expected for data)")
        weights = np.ones_like(var)

    return var, weights

In [ ]:
def signal_hists(evtdf=None,  # df with selected & reco'ed events
                 nudf=None,   # df with all MC truth
                 var_config=None,
                 return_data=False,
                 plot=True,
                 textloc=[0.05, 0.55],
                 approval="internal",
                 save_fig=False, 
                 save_name=None):

    bins = var_config.bins
    bin_centers = var_config.bin_centers
    reco_col = var_config.var_evt_reco_col

    # ===== all selected events =====
    if var_config.clip:
        var_allsel_reco, wgt_allsel_reco = get_clipped_evts(evtdf, reco_col, bins)
    else:
        var_allsel_reco, wgt_allsel_reco = get_evts(evtdf, reco_col, bins)

    # ===== true signal events =====
    evtdf_signal = evtdf[evtdf.truth.nu_categ == "CC1pi"]
    
    if var_config.clip:
        var_sel_reco, wgt_sel_reco = get_clipped_evts(evtdf_signal, reco_col, bins)
    else: 
        var_sel_reco, wgt_sel_reco = get_evts(evtdf_signal, reco_col, bins)

    # 1. Calculate standard histograms (Sum of weights)
    nevts_allsel_reco, _ = np.histogram(var_allsel_reco, weights=wgt_allsel_reco, bins=bins)
    nevts_sel_reco, _     = np.histogram(var_sel_reco,     weights=wgt_sel_reco,     bins=bins)

    # 2. Calculate Sum of Squares of Weights (Sum W^2) for error propagation
    sum_w2_allsel_reco, _ = np.histogram(var_allsel_reco, weights=wgt_allsel_reco**2, bins=bins)
    sum_w2_sel_reco, _     = np.histogram(var_sel_reco,     weights=wgt_sel_reco**2,     bins=bins)

    if return_data:
        return {
            "var_sel_reco": var_sel_reco,
            "wgt_sel_reco": wgt_sel_reco,
            "nevts_sel_reco": nevts_sel_reco,
            "sum_w2_sel_reco": sum_w2_sel_reco, # Added

            "var_allsel_reco": var_allsel_reco,
            "wgt_allsel_reco": wgt_allsel_reco,
            "nevts_allsel_reco": nevts_allsel_reco,
            "sum_w2_allsel_reco": sum_w2_allsel_reco, # Added
        }

In [ ]:
def get_univ_rates(cov_type="rate", 
                    evtdf=None, 
                    nudf=None, 
                    var_config=None, 
                    syst_name="", 
                    n_univ=100, 
                    bkgd_subtract=True,
                    plot=False):
    """
    for the GENIE uncertainty on the xsec measurement
    """

    if cov_type == "xsec":
        print("getting {} universes for {} uncertainty on the xsec".format(n_univ, syst_name))
        print(f"x-sec UNIT:{XSEC_UNIT}")
        scale_factor = XSEC_UNIT
    elif cov_type == "rate":
        print("getting {} universes for {} uncertainty on the event rate".format(n_univ, syst_name))
        scale_factor = 1.0
    else:
        raise ValueError("Invalid covariance type: {}, choose in [xsec, rate]".format(cov_type))

    bins = var_config.bins

    evtdf_signal = evtdf[evtdf.truth.nu_categ == "CC1pi"]
    # reco variable histogram, topology breakdown
    evtdf_div_topo = [evtdf[evtdf.truth.nu_categ == mode]for mode in topology_list]

    ret = signal_hists(evtdf, nudf, var_config, return_data=True, plot=plot)
    
    univ_events = []
    univ_effs   = []
    univ_smears = []

    for uidx in range(n_univ):
        syst_column = ("truth",syst_name,"univ_{}".format(uidx),"","","")
        signal_univ, _ = np.histogram(ret["var_sel_reco"], 
                                          weights=ret["wgt_sel_reco"]*evtdf_signal[syst_column],
                                          bins=bins)
        # TODO: this isn't computationally efficient, but it's useful for debugging
        # ---- uncertainty on the background rate ----
        # loop over background categories
        # + univ background - cv background
        # note: cv background subtraction cancels out with the cv background subtraction for the cv event rate. 
        #       doing it anyways for the plot of universes on background subtracted event rate.
   
        for this_evtdf in evtdf_div_topo[1:]:
            if var_config.clip:
                var, wgt = get_clipped_evts(this_evtdf, var_config.var_evt_reco_col, bins)
            else:
                var, wgt = get_evts(this_evtdf, var_config.var_evt_reco_col, bins)
                
            univ_wgt = this_evtdf[syst_column].copy()
            univ_wgt[np.isnan(univ_wgt)] = 1 ## IMPORTANT: make nan univ_wgt to 1. to ignore them
            background_cv, _   = np.histogram(var, bins=bins, weights=wgt)
            background_univ, _ = np.histogram(var, bins=bins, weights=wgt*univ_wgt)

            if bkgd_subtract:
                signal_univ += (background_univ - background_cv)
            else:
                signal_univ += background_univ


        signal_univ *= scale_factor
        univ_events.append(signal_univ)

    univ_events = np.array(univ_events)

    if bkgd_subtract:
        cv_events = ret["nevts_sel_reco"]
        cv_events *= scale_factor
    else:
        cv_events = ret["nevts_allsel_reco"]
        cv_events *= scale_factor 

    return univ_events, cv_events


In [ ]:
# ==== fractional uncertainty plot ====
def plot_frac_unc(frac_unc_named_list,  # Expecting [(unc_array, "name"), ...]
                  var_config, 
                  plot_labels=["", "", ""],
                  approval="internal",
                  plot=True,
                  save_fig=False, 
                  save_name=None,
                  fig_ext=".pdf"):

    fig, ax = plt.subplots(figsize=(8, 6))
    
    max_val = 0
    for fidx, (frac_unc, label_name) in enumerate(frac_unc_named_list):
        color = "C{}".format(fidx)
        if len(frac_unc_named_list) == 1:
            color = "black"
        if label_name == "Total":
            color = "black"
            
        # Plotting in percent [%]
        ax.hist(var_config.bin_centers, bins=var_config.bins, 
                weights=frac_unc * 100, histtype="step", 
                color=color, linewidth=2, label=label_name)
        
        # Track max (account for the *100 scaling)
        current_max = np.nanmax(np.nan_to_num(frac_unc * 100, nan=0, posinf=0))
        if current_max > max_val:
            max_val = current_max

    # --- Axis Formatting ---
    ax.set_xlim(var_config.bins[0], var_config.bins[-1])
    ax.set_ylabel("Fractional Uncertainty [%]")
    
    if max_val == 0: max_val = 10.0 # Default to 10% if empty
    ax.set_ylim(0, max_val * 1.2) # Use 1.2 to leave room for the legend
    
    ax.set_title(plot_labels[2])
    ax.grid(True, linestyle='--', alpha=0.6)
    
    # Add the legend!
    ax.legend(loc='upper right', frameon=True)

    add_approval_text(approval, 0.02, 0.98, "left")
               
    if save_fig and save_name:
        plt.savefig(f"{save_name}{fig_ext}", bbox_inches='tight')

    if plot:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
def get_covariance_matrix(univ_events, cv_events):

    n_univ, n_bins = univ_events.shape

    cov_frac = np.zeros((n_bins, n_bins))
    cov = np.zeros((n_bins, n_bins))

    for uidx in range(n_univ):
        for i in range(n_bins):
            for j in range(n_bins):

                nom_i = cv_events[i]
                nom_j = cv_events[j]

                univ_i = univ_events[uidx, i]
                univ_j = univ_events[uidx, j]

                cov_entry = (univ_i - nom_i) * (univ_j - nom_j)

                frac_cov_entry = (
                    ((univ_i - nom_i) / nom_i) *
                    ((univ_j - nom_j) / nom_j)
                )

                cov[i, j] += cov_entry
                cov_frac[i, j] += frac_cov_entry

    cov /= n_univ
    cov_frac /= n_univ

    cov = np.nan_to_num(cov, nan=0.0)
    cov_frac = np.nan_to_num(cov_frac, nan=0.0)

    # 🔥 CLIP FRACTIONAL COVARIANCE
    cov_frac = np.clip(cov_frac, -25.0, 25.0)

    corr = np.zeros_like(cov)

    for i in range(len(cv_events)):
        for j in range(len(cv_events)):
            denom = np.sqrt(cov[i, i]) * np.sqrt(cov[j, j])
            if denom != 0:
                corr[i, j] = cov[i, j] / denom
            else:
                corr[i, j] = 0.0

    return {
        "cov_frac": cov_frac,
        "cov": cov,
        "corr": corr,
    }

In [ ]:

def variation_hists(evtdfs=None, var_name=None, clip = False, 
                    nevts_list=None,
                    datadf=None,
                    bins=None,
                    var_colors=None, var_labels=None,
                    plot_labels=["", "", ""],
                    vline = None,
                    textloc=[0.05, 0.55],
                    approval="internal",
                    plot=True,
                    save_fig=False, save_name=None): 

    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    if evtdfs is not None:
        n_vars = len(evtdfs)
    elif nevts_list is not None:
        n_vars = len(nevts_list)
    else:
        raise ValueError("Either evtdfs or nevts_list must be provided")

    # get distribution from dfs
    if evtdfs is not None:
        vardfs, wgtdfs = [], []
        nevts_list = []
        mc_stat_err_list = []
        for df in evtdfs:
            if clip:
                vardf, wgtdf = get_clipped_evts(df, var_name, bins)
            else:
                vardf, wgtdf = get_evts(df, var_name, bins)
                
           
            vardfs.append(vardf)
            wgtdfs.append(wgtdf)


        # for sidx in range(n_vars):
            nevts, _ = np.histogram(vardf, bins=bins, weights=wgtdf)
            total_mc_err2, _ = np.histogram(vardf, bins=bins, weights=wgtdf**2)
            mc_stat_err = np.sqrt(total_mc_err2)
            nevts_list.append(nevts)
            mc_stat_err_list.append(mc_stat_err)

    return nevts_list

# Version that imports the systematics for the final vars

In [ ]:
def filter_df(df, mask_key, cut_mask, config):
    """Applies cut masks, extra masks, and slice grouping."""
    filtered = df[cut_mask & mask_dict[config.extra_mask](df)]
    if config.first_per_slice:
        filtered = filtered.groupby(level=SLICE_LEVELS, sort=False).first()
    return filtered

In [ ]:
import os, shutil, tarfile
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


config_p_pi_final = FullHistogramConfig(
    file_name = "pion_p",      
    var_evt_reco_col=('slc','measure_var','TLE_p_pi','','',''),
    truth_column = pion_p_type_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins= np.array([0.13, 0.218, 0.296,0.415,1]),
    xlabel=r'Pion candidate P [GeV]',
    ylabel=slices_y_label
)


# --- Configuration & Setup ---
config_vec = final_var_configs
config_vec = [config_p_pi_final]
FULL_SYST = True

if FULL_SYST:
    parent_path = "/exp/sbnd/data/users/lpelegri/Graphs/FinalMCDataCompGraphsFullErr"
else:
    parent_path = "/exp/sbnd/data/users/lpelegri/Graphs/FinalMCDataCompGraphs"

'''
if plot_sideband:
    parent_path += "sideband"
'''

good_base_path, bad_base_path = os.path.join(parent_path, "good"), os.path.join(parent_path, "bad")
tar_output_path = f"{parent_path}.tar"

CHI2_THRESHOLD = 5
PLOT_IND_UNCR = True
WGT_COL = ('slc', 'wgt', '', '', '', '')
SLICE_LEVELS = ['__ntuple', 'entry', 'rec.slc..index']

# Color codes
RED, RESET = "\033[31m", "\033[0m"

# Clean workspace
for path in [parent_path, tar_output_path]:
    if os.path.exists(path):
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
os.makedirs(good_base_path, exist_ok=True); os.makedirs(bad_base_path, exist_ok=True)

file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices_rate_no_bkg_substracted"

# Load Systematic Dictionaries
flux_syst = np.load(file_dir + "/extended_flux_syst_dict_rate_ar23p.npz")
g4_syst = np.load(file_dir + "/extended_g4_syst_dict_rate_ar23p.npz")
genie_syst = np.load(file_dir + "/extended_genie_syst_dict_xsec_ar23p.npz")


file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices"
cosmics_syst = np.load(file_dir + "/cosmics_syst_dict.npz")
detvar_syst = np.load(file_dir + "/detvar_syst_dict.npz")
mcstat_syst = np.load(file_dir + "/mcstat_syst_dict_ar23p.npz")

'''
flux_syst = np.load(file_dir + "/extended_flux_syst_dict.npz")
g4_syst = np.load(file_dir + "/extended_g4_syst_dict.npz")
genie_syst = np.load(file_dir + "/extended_genie_xsec_syst_dict.npz")
mcstat_syst = np.load(file_dir + "/mcstat_syst_dict.npz")
cosmics_syst = np.load(file_dir + "/cosmics_syst_dict.npz")
detvar_syst = np.load(file_dir + "/detvar_syst_dict.npz")
'''
pot_frac_unc = 0.02
ntargets_frac_unc = 0.01


GENIE_TAGS = [False, True]

# --- Main Analysis Loop ---
for PLOT_GENIE_CATEG in GENIE_TAGS:
    for config in config_vec:
        # Determine cut range
        idx_range = sorted([cuts.index(config.start_cut), cuts.index(config.end_cut)])
        selected_cuts = cuts[idx_range[0] : idx_range[1] + 1]
        '''
        if PLOT_GENIE_CATEG:
            config.truth_column = ('truth','genie_categ','','','','')
        else:
            config.truth_column = ('truth','nu_categ','','','','')
        '''
        for cut in selected_cuts: 
            print(f"--- Processing Cut: {cut} ---")
            
            # 1. Prepare Dataframes
            curr_mc = filter_df(mc_evt_df, config.extra_mask, mc_cumulative_masks[cut], config)
            curr_data = filter_df(data_evt_df, config.extra_mask, data_cumulative_masks[cut], config)
            
            # Determine number of bins from config to initialize matrices
            # (This avoids the IndexError by matching the dimension of the loaded matrices)
            n_bins = len(config.bins) - 1
            bin_centers = (config.bins[:-1] + config.bins[1:]) / 2.
    
            # 2. Systematic Matrix Aggregation
            # We start with MC Statistics as the baseline matrix
            total_cov_frac = mcstat_syst[config.file_name].copy()
            frac_unc_list = [(np.sqrt(np.diag(total_cov_frac)), "MCStat")]
            
            if FULL_SYST:
                # Binned Systematics (Matrices)
                systs = [flux_syst, g4_syst, genie_syst, detvar_syst, cosmics_syst]
                syst_names = ["Flux", "G4", "Genie", "Detector", "Cosmic"]
    
                for name, syst_dict in zip(syst_names, systs):
                    matrix = syst_dict[config.file_name]
                    total_cov_frac += matrix # Matrix addition preserves correlations
                    frac_unc_list.append((np.sqrt(np.diag(matrix)), name))
    
                # Flat Systematics (Normalization)f
                # These are 100% correlated across all bins
                flat_systs = [pot_frac_unc, ntargets_frac_unc]
                flat_names = ["POT", "Ntargets"]
    
                for name, val in zip(flat_names, flat_systs):
                    # Create a matrix where every element is (sigma_flat)^2
                    flat_matrix = np.full((n_bins, n_bins), val**2)
                    total_cov_frac += flat_matrix
                    frac_unc_list.append((val * np.ones(n_bins), name))
    
            # 3. Final Uncertainty Vector for Plotting
            total_uncertainty_vec = np.sqrt(np.diag(total_cov_frac))
            final_plot_list = [(total_uncertainty_vec, "Total")] + frac_unc_list
            plot_frac_unc(final_plot_list, config)
            
            # 4. Convert Fractional Covariance to Absolute Covariance for Chi2
            # Use the MC counts for the current cut to scale the matrix
            # 'ret' logic usually comes from the plotter; ensure you have mc_counts here
            mc_counts, _ = np.histogram(curr_mc[config.var_evt_reco_col], bins=config.bins, weights=curr_mc[WGT_COL])
            
            total_cov = cov_from_fraccov(total_cov_frac, mc_counts)
    
            # 5. Plotting and Chi2 Calculation
            fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=curr_mc, data_df=curr_data, config=config,
                cov_frac_matrix=total_cov_frac, cov_matrix=total_cov,
                title=f"{cut} - {config.file_name}", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=FULL_SYST, symmetric_ratio = True, divide_by_bin_width = True
            )
    
            # 6. File Management
            status_path = good_base_path if red_chi2 < CHI2_THRESHOLD else bad_base_path
            save_dir = os.path.join(status_path, config.file_name)
            os.makedirs(save_dir, exist_ok=True)

            topology_name = "topology"
            if PLOT_GENIE_CATEG:
                topology_name = "genie"
            
            f_name = f"cut_{cut}_{config.file_name}_{topology_name}.pdf"
            fig.savefig(os.path.join(save_dir, f_name), format='pdf', bbox_inches='tight')
            plt.close(fig)
    
            color = RED if red_chi2 > CHI2_THRESHOLD else ""
            print(f"{color}Saved {f_name} (Chi2: {red_chi2:.2f}){RESET}")

# --- Archive Results ---
print(f"Archiving to {tar_output_path}...")
with tarfile.open(tar_output_path, "w:gz") as tar:
    tar.add(parent_path, arcname=os.path.basename(parent_path))
print("Done!")